# 🔴 Solution: Grouped Query Attention（面试版）

## 📋 题目描述

**难度：** Hard

实现分组查询注意力（GQA）。

GQA 使用比查询头更少的 KV 头，每个 KV 头在一组查询头之间共享，在保持质量的同时减少 KV 缓存大小。

**签名:** `GroupQueryAttention(d_model, num_heads, num_kv_heads)`

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (B, S, d_model)

**返回:** 注意力输出 (B, S, d_model)

**约束:**
- W_k/W_v 投影到 `num_kv_heads * d_k` 维
- 使用 `repeat_interleave` 扩展 KV 头以匹配查询头数
- 当 `num_kv_heads == num_heads` 时退化为标准 MHA

**提示：** 1. `W_q` → `(B,H,S,d_k)`，`W_k`/`W_v` → `(B,KV,S,d_k)`
2. `K = K.repeat_interleave(H//KV, dim=1)` 扩展 KV 头
3. 缩放点积注意力 → reshape 为 `(B,S,d_model)`


In [ ]:
import torch
import math

In [ ]:
# ✅ INTERVIEW

def grouped_query_attention(x: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, num_heads: int, num_kv_heads: int, mask: torch.Tensor = None) -> torch.Tensor:
    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # ---- Step 1: 投影 ----
    Q = x @ W_q  # [B, S, d_model]
    K = x @ W_k  # [B, S, num_kv_heads * d_k]
    V = x @ W_v  # [B, S, num_kv_heads * d_k]

    # ---- Step 2: 拆分 ----
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)
    K = K.view(batch_size, seq_len, num_kv_heads, d_k).transpose(1, 2)
    V = V.view(batch_size, seq_len, num_kv_heads, d_k).transpose(1, 2)

    # ---- Step 3: KV 扩展 ----
    # repeat_interleave：将 KV heads 复制以匹配 query heads
    # 关键：不是 expand，需要实际复制数据
    n_rep = num_heads // num_kv_heads
    K = K.repeat_interleave(n_rep, dim=1)
    V = V.repeat_interleave(n_rep, dim=1)

    # ---- Step 4: 注意力 ----
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # 手写 softmax
    scores_max = scores.max(dim=-1, keepdim=True).values
    exp_scores = torch.exp(scores - scores_max)
    attn = exp_scores / exp_scores.sum(dim=-1, keepdim=True)

    out = attn @ V

    # ---- Step 5: 合并 + 投影 ----
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    return out @ W_o

# 面试追问：
# Q: GQA vs MHA vs MQA？
# A: MHA 每 head 独立 KV；GQA 多 head 共享 KV；MQA 所有 head 共享 1 组 KV
# Q: 为什么用 GQA？
# A: 减少 KV cache 大小，推理更快，效果接近 MHA

In [ ]:
gqa = GroupQueryAttention(32, 8, 2)
print('Output:', gqa.forward(torch.randn(1, 4, 32)).shape)

## 📝 核心思路总结

1. **GQA 核心**：多个 query head 共享 KV head，减少 KV cache
2. **repeat_interleave**：扩展 KV 以匹配 query head 数
3. **GQA = MHA + MQA 的折中**：平衡效果和效率
4. **推理优化**：KV cache 减少 num_heads/num_kv_heads 倍